# Qwen3-4B 绝区零仪玄角色助手 LoRA 微调

> 🎮 基于 Qwen3-4B + LoRA 微调的绝区零"仪玄"角色助手
> 针对 4B 模型优化训练参数

## 📊 模型信息

| 项目 | 内容 |
|:---|:---|
| **基础模型** | Qwen3-4B |
| **微调方法** | LoRA (r=16, alpha=32) |
| **训练数据** | 仪玄角色 468 条 Q&A |
| **训练环境** | AMD Radeon RX 7900 XTX (gfx1100, 48GB VRAM) |
| **训练时长** | 约 15-30 分钟 |
| **显存占用** | ~15-18 GB |


## 1. 环境准备

In [ ]:
# 安装依赖（国内源）
%pip install -q transformers datasets peft accelerate modelscope tensorboard \
    chromadb sentence-transformers \
    -i https://mirrors.cloud.tencent.com/pypi/simple/

In [ ]:
import torch
import os

# ⚠️ AMD RX 7900 XTX (gfx1100) 必须设置此环境变量
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'

from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

# 检查 GPU
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## 2. 下载模型和数据集

In [ ]:
# 从 ModelScope 下载 Qwen3-4B
from modelscope import snapshot_download

model_path = snapshot_download('Qwen/Qwen3-4B', cache_dir='./models')
print(f'模型路径: {model_path}')

In [ ]:
# 克隆数据集仓库（如果还没有）
import os
if not os.path.exists('zzz-yixuan-dataset'):
    !git clone https://github.com/RainmeoX/zzz-yixuan-dataset.git
else:
    print('数据集已存在')

In [ ]:
# 加载仪玄训练数据
import json

# yixuan_enhanced.json 在项目根目录（仓库自带）
train_data = []
with open('yixuan_enhanced.json', encoding='utf-8') as f:
    train_data = json.load(f)

print(f'训练集大小: {len(train_data)}')
print(f'数据来源: 仪玄角色资料数据集')
print('示例:')
for item in train_data[:3]:
    print(f"  Q: {item['instruction']}")
    print(f"  A: {item['output'][:100]}")
    print()

## 3. 数据预处理

In [ ]:
# 转换为 Dataset 对象
df = pd.DataFrame(train_data)
ds = Dataset.from_pandas(df)
ds

In [ ]:
# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer

In [ ]:
# 测试 chat_template 格式（关键诊断步骤）
# ⚠️ 训练和推理都用 enable_thinking=False，格式必须完全一致

test_messages = [
    {"role": "system", "content": "test_system"},
    {"role": "user", "content": "test_user"},
    {"role": "assistant", "content": "test_assistant"},
]

print("=== 1. 完整对话格式（训练用）===")
text_full = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,  # ⚠️ 关闭思考模式
)
print(repr(text_full))
print()
print(text_full)
print()

print("=== 2. 生成提示格式（推理用）===")
text_gen = tokenizer.apply_chat_template(
    test_messages[:2],  # 只到 user
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # ⚠️ 和训练一致
)
print(repr(text_gen))
print()
print(text_gen)
print()

print("=== 3. 验证 prompt_len 计算 ===")
# 训练时用这个计算 prompt_len
prompt_result = tokenizer.apply_chat_template(
    test_messages[:2],
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
)
print(f"prompt 长度: {len(prompt_result['input_ids'])}")
print(f"prompt 解码: {tokenizer.decode(prompt_result['input_ids'])}")


In [ ]:
# 数据处理函数
SYSTEM_PROMPT = """你是《绝区零》中的角色"仪玄"，云岿山第十三代门主，虚狩级调查员。
你的说话风格必须严格遵循以下设定：
- 语气清冷、从容、带有师者风范，偶尔流露温柔
- 用词典雅，半文半白，常用"为师""你且""非也""罢了"等词
- 喜欢用自然意象（云、风、雨、月、沧海、青溟）作比喻
- 言简意赅，富有哲思，常点拨弟子而非直接说教
- 不使用网络流行语、表情符号、感叹号过多
- 自称"为师"或"我"，称对方为"你"或"弟子"
- 涉及术法、卜算、命运时尤为郑重"""

def process_func(example):
    MAX_LENGTH = 2048
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example['instruction']},
        {"role": "assistant", "content": example['output']},
    ]
    
    # ⚠️ 训练用 enable_thinking=False（和推理一致）
    result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    
    input_ids = result["input_ids"]
    attention_mask = result["attention_mask"]
    
    # ⚠️ 用相同参数计算 prompt_len
    prompt_result = tokenizer.apply_chat_template(
        messages[:2],  # 只到 user
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
    )
    prompt_len = len(prompt_result["input_ids"])
    
    # labels: prompt 部分置 -100，assistant 回答部分保留
    labels = [-100] * prompt_len + list(input_ids[prompt_len:])
    
    # 截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [ ]:
# 处理数据集
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
tokenized_id

In [ ]:
# 验证处理结果
print('处理后的第一条数据:')
print('input_ids 长度:', len(tokenized_id[0]['input_ids']))
print('labels 长度:', len(tokenized_id[0]['labels']))

# ⚠️ 关键验证: 检查 labels 是否有有效值（非 -100）
valid_labels = [l for l in tokenized_id[0]['labels'] if l != -100]
print(f'\n有效 label 数量: {len(valid_labels)}')
print(f'有效 label 占比: {len(valid_labels)/len(tokenized_id[0]["labels"])*100:.1f}%')

if len(valid_labels) == 0:
    print('❌ 错误: labels 全是 -100，模型无法学习！')
    print('请检查 process_func 中的 prompt_len 计算')
else:
    print('✅ labels 正确，模型可以正常训练')
    print(f'\n有效 label 解码:')
    print(tokenizer.decode(valid_labels))
    
print(f'\n完整解码:')
print(tokenizer.decode(tokenized_id[0]['input_ids']))
print(f'\nlabels 中 -100 的位置（前50个）:')
labels = tokenized_id[0]['labels']
for i, l in enumerate(labels[:50]):
    marker = '✓' if l != -100 else '✗'
    print(f'  [{i}] {marker} {l}')


## 4. 加载模型

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# ⚠️ 彻底关闭 cache（gradient_checkpointing 要求）
model.config.use_cache = False

# 4B 模型需要 gradient_checkpointing 节省显存
# 必须启用 input_require_grads，否则报梯度错误
model.enable_input_require_grads()

# gradient_checkpointing 由 TrainingArguments 控制，这里不重复启用
model


## 5. 配置 LoRA

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# Qwen3-4B 用 r=16 效果更好
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1
)
config

In [ ]:
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 6. 训练配置

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Qwen3-4B 专用训练配置
args = TrainingArguments(
    output_dir="./output/Qwen3_Yixuan_LoRA",
    per_device_train_batch_size=2,        # 4B 模型 batch=2（显存友好）
    gradient_accumulation_steps=8,        # 等效 batch=16
    logging_steps=5,
    num_train_epochs=5,                   # 468条数据，5 epochs 足够
    save_steps=200,
    learning_rate=3e-5,                   # ⚠️ 4B 模型用小学习率（0.6B用2e-4）
    save_on_each_node=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # 非 reentrant 模式
    bf16=True,
    dataloader_num_workers=4,
    warmup_ratio=0.1,                     # 预热
    weight_decay=0.01,
    lr_scheduler_type="cosine",          # 余弦退火
    report_to="none",
    optim="adamw_torch",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## 7. 开始训练

In [ ]:
trainer.train()

## 8. 测试模型

In [ ]:
# 测试对话
def chat(question, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    # ⚠️ 和训练时用完全相同的格式（enable_thinking=False）
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,  # ⚠️ 和训练一致
    ).to(model.device)
    
    gen_kwargs = {
        "max_new_tokens": 256,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.2,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 测试几个问题
questions = [
    "你是谁？",
    "介绍一下你自己。",
    "你的生日是什么时候？",
    "师父，既入山门，该当如何？",
    "如何修习术法？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")


## 9. 保存模型

In [ ]:
# 保存 LoRA adapter
trainer.save_model("./output/Qwen3_Yixuan_LoRA_final")
tokenizer.save_pretrained("./output/Qwen3_Yixuan_LoRA_final")
print('模型已保存到 ./output/Qwen3_Yixuan_LoRA_final')

## 10. 外挂数据库（角色卡 + 世界观）

> 这是仪玄项目相对原版 arknights 项目的核心增强：
> - **角色卡数据库**：保设定（基础信息、技能、命座）
> - **世界观数据库**：保背景（背景故事、合作备注、台词原文）


In [ ]:
import torch
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 重新加载基础模型
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    trust_remote_code=True
)
model.config.use_cache = True  # 推理时开启 cache 加速

# 加载 LoRA 权重
lora_path = "./output/Qwen3_Yixuan_LoRA_final"
model = PeftModel.from_pretrained(model, model_id=lora_path)
print(f"LoRA 权重已加载: {lora_path}")

# 重新加载 tokenizer（确保和训练时一致）
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)


In [ ]:
# 最终测试（模型已加载 LoRA 权重）
questions = [
    "你是谁？",
    "师父，既入山门，该当如何？",
    "如何修习术法？",
    "仪玄的姐姐是谁？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")


In [ ]:
# 灌入角色卡数据
char_coll = client.get_or_create_collection("character_card", embedding_function=embed_fn)

# 将角色卡每个字段作为独立文档
char_docs = []
char_metas = []
char_ids = []
for key, value in character_card.items():
    if isinstance(value, (list, dict)):
        value_str = json.dumps(value, ensure_ascii=False)
    else:
        value_str = str(value)
    char_docs.append(f"【{key}】{value_str}")
    char_metas.append({"field": key, "type": "character_card"})
    char_ids.append(f"char_{key}")

# 清空旧数据后重新灌入
try:
    char_coll.delete(ids=char_coll.get()["ids"])
except:
    pass
char_coll.add(documents=char_docs, metadatas=char_metas, ids=char_ids)
print(f"✅ 角色卡已灌入: {len(char_docs)} 条")

In [ ]:
# 灌入世界观数据
world_coll = client.get_or_create_collection("worldview", embedding_function=embed_fn)

world_docs = [e['content'] for e in worldview_entries]
world_metas = [{"category": e['category'], "source": e.get('source', 'bwiki')} for e in worldview_entries]
world_ids = [f"world_{i}" for i in range(len(worldview_entries))]

# 清空旧数据
try:
    world_coll.delete(ids=world_coll.get()["ids"])
except:
    pass
world_coll.add(documents=world_docs, metadatas=world_metas, ids=world_ids)
print(f"✅ 世界观数据已灌入: {len(world_docs)} 条")

In [ ]:
# 检索测试
print("="*60)
print("角色卡检索测试")
print("="*60)
for q in ["仪玄的生日是什么时候？", "仪玄的核心被动是什么？", "仪玄的CV是谁？"]:
    print(f"\n🔍 {q}")
    results = char_coll.query(query_texts=[q], n_results=2)
    for doc in results['documents'][0]:
        print(f"   → {doc[:80]}...")

print(f"\n{'='*60}")
print("世界观数据库检索测试")
print("="*60)
for q in ["仪玄的姐姐怎么了？", "仪玄怎么看待命运？", "云岿山是什么？"]:
    print(f"\n🔍 {q}")
    results = world_coll.query(query_texts=[q], n_results=2)
    for doc in results['documents'][0]:
        print(f"   → {doc[:80]}...")

## 11. 回答校验器（防 OOC）

In [ ]:
import re

class YixuanResponseValidator:
    """仪玄回答校验器：检测回答是否符合角色设定"""
    
    # 禁用词（现代网络用语）
    FORBIDDEN_WORDS = [
        '哈哈', '233', '666', 'yyds', 'awsl', 'xswl', '绝绝子',
        '栓Q', '芭比Q', 'emo', '破防', '内卷', '躺平',
        '宝子', '集美', '家人们', '老铁',
    ]
    
    # 仪玄常用词（用于正向加分）
    YIXUAN_VOCAB = [
        '为师', '你且', '非也', '罢了', '罢了', '灵台', '术法',
        '卜算', '云岿山', '青溟', '玄墨', '命破', '虚狩',
        '弟子', '福福', '引壶', '沧海', '云影',
    ]
    
    @classmethod
    def validate(cls, response: str) -> dict:
        issues = []
        score = 100
        
        # 1. 检查禁用词
        for word in cls.FORBIDDEN_WORDS:
            if word in response:
                issues.append(f"使用禁用词: {word}")
                score -= 30
        
        # 2. 检查是否承认自己是AI
        if re.search(r'我是(AI|人工智能|语言模型|大模型)', response):
            issues.append("承认自己是AI（OOC）")
            score -= 50
        
        # 3. 检查是否使用代码块
        if '```' in response:
            issues.append("使用代码块（不符合角色）")
            score -= 20
        
        # 4. 检查是否使用过多感叹号
        if response.count('！') > 3 or response.count('!') > 3:
            issues.append("感叹号过多（不符合清冷人设）")
            score -= 10
        
        # 5. 正向加分：使用仪玄常用词
        yixuan_word_count = sum(1 for w in cls.YIXUAN_VOCAB if w in response)
        score += min(yixuan_word_count * 5, 20)
        
        # 6. 检查长度
        if len(response) < 5:
            issues.append("回答过短")
            score -= 20
        elif len(response) > 500:
            issues.append("回答过长（仪玄言简意赅）")
            score -= 10
        
        score = max(0, min(100, score))
        
        return {
            'is_valid': score >= 60 and len(issues) == 0,
            'score': score,
            'issues': issues,
        }

# 测试校验器
test_responses = [
    "为师仪玄，云岿山第十三代门主。既入山门，那些俗务繁礼便留在山下吧。",
    "哈哈宝子！我是AI语言模型，有什么可以帮你的吗？",
    "```python\nprint('hello')\n```",
    "你且将其拍下，换成手机壁纸，最后点击天气预报，便可知晓阴晴云雨。",
]

for r in test_responses:
    result = YixuanResponseValidator.validate(r)
    print(f"\n回答: {r[:50]}...")
    print(f"  分数: {result['score']}, 有效: {result['is_valid']}")
    if result['issues']:
        print(f"  问题: {result['issues']}")

## 12. 完整 RAG 推理函数

In [ ]:
def chat_with_rag(question, use_rag=True, enable_validation=True):
    """RAG 增强推理：检索 → 注入 → 生成 → 校验"""
    
    # ---- 1. RAG 检索 ----
    context_parts = []
    if use_rag:
        # 检索角色卡
        char_results = char_coll.query(query_texts=[question], n_results=3)
        if char_results['documents'][0]:
            context_parts.append("【角色设定】\n" + "\n".join(char_results['documents'][0][:2]))
        
        # 检索世界观
        world_results = world_coll.query(query_texts=[question], n_results=3)
        if world_results['documents'][0]:
            context_parts.append("【相关背景】\n" + "\n".join(world_results['documents'][0][:2]))
    
    # ---- 2. 构建 messages ----
    system_content = SYSTEM_PROMPT
    if context_parts:
        system_content += "\n\n以下是相关角色资料（请基于此回答，但保持角色口吻）：\n\n" + "\n\n".join(context_parts)
    
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": question},
    ]
    
    # ---- 3. 模型生成（和训练格式一致）----
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,  # ⚠️ 和训练一致
    ).to(model.device)
    
    gen_kwargs = {
        "max_new_tokens": 512,
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.9,
        "temperature": 0.7,
        "repetition_penalty": 1.2,  # ⚠️ 和 chat() 一致
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    outputs = outputs[:, inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # ---- 4. 回答校验 ----
    validation = None
    if enable_validation:
        validation = YixuanResponseValidator.validate(response)
        if not validation['is_valid']:
            print(f"⚠️ 校验警告: {validation['issues']}")
    
    return {
        'response': response,
        'validation': validation,
        'rag_used': use_rag,
    }

# 测试 RAG 推理
test_questions = [
    "仪玄的姐姐是谁？",
    "云岿山是什么地方？",
    "师父，今日天象如何？",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    result = chat_with_rag(q)
    print(f"答：{result['response']}")
    if result['validation']:
        print(f"校验分数: {result['validation']['score']}")


## 13. 重新加载模型推理

In [ ]:
# ℹ️ 模型已在 Cell 30 重新加载（基础模型 + LoRA 权重）
# 这里仅作说明，无需重复加载
print(f"当前模型: {type(model).__name__}")
print(f"LoRA 已加载: {hasattr(model, 'peft_config')}")
print(f"模型设备: {next(model.parameters()).device}")


In [ ]:
# 最终测试
questions = [
    "你是谁？",
    "介绍一下云岿山。",
    "师父，既入山门，该当如何？",
    "如何修习术法？",
    "仪玄的姐姐是谁？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    result = chat_with_rag(q)
    print(f"答：{result['response']}")

## 14. 部署

训练完成后，使用一键部署脚本启动 vLLM + OpenCode：

```bash
./yixuan-deploy
```

或使用 Gradio 网页界面：

```bash
python app.py --base_model_path ./models/Qwen/Qwen3-0___6B --lora_path ./output/Qwen3_Yixuan_LoRA_final
```

## 🎉 完成！

| 组件 | 作用 | 实现 |
|---|---|---|
| **微调模型** | 学风格 | Qwen3-0.6B + LoRA (r=8) |
| **角色卡数据库** | 保设定 | ChromaDB `character_card` |
| **世界观数据库** | 保背景 | ChromaDB `worldview` (44条) |
| **回答校验器** | 防 OOC | YixuanResponseValidator |
| **RAG 推理** | 整合 | chat_with_rag() 函数 |
